In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from gtda.pipeline import Pipeline
from gtda.time_series import Resampler
from gtda.diagrams import PersistenceEntropy, Scaler, HeatKernel, BettiCurve
import numpy as np
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters, TakensEmbedding
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.metaestimators import CollectionTransformer

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


We start by analyzing the public data set and looking at the normal operations time series, to set a benchmark for the analysis of well events (abnormal operations). 

In [ ]:
sample_data_df = pd.read_csv(DATA_3W_NORMAL + "/WELL-00008_20170630130246.csv")

In [ ]:
sample_data_df = pd.read_csv(DATA_3W_NORMAL + "/WELL-00008_20170914020321.csv")

In [ ]:
fig, ts = plt.subplots(5,figsize=(10,20),sharex = True)

fig.suptitle('Normal operations dataset #random')
ts[4].set_xlabel('time')
ts[0].set_ylabel("P-PDG")
ts[1].set_ylabel('P-TPT')
ts[2].set_ylabel('T-TPT')
ts[3].set_ylabel("P-MON-CKP")
ts[4].set_ylabel('T-JUS-CKP')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(sample_data_df.index, sample_data_df["P-PDG"])
ts[1].plot(sample_data_df.index, sample_data_df["P-TPT"])
ts[2].plot(sample_data_df.index, sample_data_df["T-TPT"])
ts[3].plot(sample_data_df.index, sample_data_df["P-MON-CKP"])
ts[4].plot(sample_data_df.index, sample_data_df["T-JUS-CKP"])

It looks like the most information is stored in the TPT (bottomhole) pressure and temperature time series. We start from the Pressure. 

## Analysis of Normal Operations: P- TPT

In [ ]:
find_shortest_file(DATA_3W_NORMAL, "P-TPT")

The time series are pretty large, so we rescale them to a uniform resampling at 3000 points, to be managenable for computing their embedding and persistence diagrams. We keep 3000 points for displying point clouds and 1000 for diagrams.

In [ ]:
normal_signals, normal_df = read_files(DATA_3W_NORMAL, "P-TPT",3000)

In [ ]:
normal_df

In [ ]:
for i in range(len(normal_df)):
    plt.figure()
    tmp = str(i)
    #plt.title.set_title(tmp)
    plt.gca().set_title(tmp)
    plt.plot(normal_df[i])
    # Show/save figure as desired.
plt.show()

In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

from random import gauss
from random import seed
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.gofplots import qqplot

freq_units = 0.1/len(normal_df[566])
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('steady state BHP stats')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('BHP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP')
data[2].set_ylabel('count')

data[1].set_xlim(2,1000)
data[1].set_yscale('log')
data[1].set_xscale('log')

ss_BHP = normal_df[566]/100000 # from Pa to Bar

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
# write file for making figures  
file = open("normal_3W.dat", "w")
for i in ss_BHP:
    tmp = str(i) + "\n"
    file.writelines(tmp)
file.close()

In [ ]:
qqplot(ss_BHP, line='s')

In [ ]:
import statsmodels.api as sm

fig = plt.figure(figsize=(12,15))
ax1 = fig.add_subplot(411)
fig = sm.graphics.tsa.plot_acf(ss_BHP.values.squeeze(), lags=500, ax=ax1)
ax2 = fig.add_subplot(412)
fig = sm.graphics.tsa.plot_acf(ss_BHP.values.squeeze(), lags=120, ax=ax2)
ax3 = fig.add_subplot(413)
fig = sm.graphics.tsa.plot_pacf(ss_BHP, lags=500, ax=ax3)
ax4 = fig.add_subplot(414)
fig = sm.graphics.tsa.plot_pacf(ss_BHP, lags=40, ax=ax4)

In [ ]:
max_time_delay = 20 
max_embedding_dimension = 10
stride = 3

signal = normal_df[566]/100000 # from Pa to Bar

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_time_delay
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_embedded_pca = pca.fit_transform(y_embedded)

norm = plot_point_cloud(y_embedded_pca)
norm

In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_embedded[None,:,:])

In [ ]:
PE_BHP = PersistenceEntropy()
features = PE_BHP.fit_transform(PerDiagram)
features 

In [ ]:
PE_BHP = PersistenceEntropy(normalize=True)
features = PE_BHP.fit_transform(PerDiagram)
features 

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in PerDiagram: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))

print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

In [ ]:
Betti = BettiCurve()
Betti_wn = Betti.fit_transform_plot(PerDiagram)

In [ ]:
# write file for making figures  
file1 = open("norm3W_Pers_H0.dat", "w")
file2 = open("norm3W_Pers_H1.dat", "w")
for i in PerDiagram[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

Some of these time series are not stationary. I convert Pa to Bar and remove those with a drift higher than 5 Bar.

In [ ]:
naughty_list = []
for timeserie in normal_df:
    if abs(normal_df[timeserie].iloc[-1]-normal_df[timeserie].iloc[0]) > 500000:
        print(timeserie, normal_df[timeserie].iloc[-1], normal_df[timeserie].iloc[0])
        naughty_list.append(timeserie)

In [ ]:
PatoBar = 1/100000
normal_df = normal_df.apply(lambda x: x*PatoBar) 
normal_signals = normal_signals * PatoBar

Removing from the steady state dataframe, those timeseries that have a deviation of more than 5 Bar

In [ ]:
normal_df = normal_df.drop(naughty_list, axis=1)
normal_df

In [ ]:
tmp = normal_df[normal_df.index % 3 == 0]  # Excludes every 3rd row starting from 0
tmp

In [ ]:
fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))

from random import gauss
from random import seed
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.gofplots import qqplot

freq_units = 0.1/len(normal_df[25])
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('steady state BHP stats')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('BHP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP')
data[2].set_ylabel('count')

data[1].set_xlim(2,1000)
data[1].set_yscale('log')
data[1].set_xscale('log')

ss_BHP = normal_df[25]

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
# normality test
from scipy.stats import anderson

result = anderson(ss_BHP)
print('Statistic: %.3f' % result.statistic)
p = 0
for i in range(len(result.critical_values)):
	sl, cv = result.significance_level[i], result.critical_values[i]
	if result.statistic < result.critical_values[i]:
		print('%.3f: %.3f, data looks normal (fail to reject H0)' % (sl, cv))
	else:
		print('%.3f: %.3f, data does not look normal (reject H0)' % (sl, cv))

In [ ]:
max_time_delay = 80 
max_embedding_dimension = 25
stride = 3

signal = normal_df[25]

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_time_delay
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_embedded_pca = pca.fit_transform(y_embedded)

norm = plot_point_cloud(y_embedded_pca)
norm

In [ ]:
homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

y_embedded_reshaped = y_embedded.reshape(1, *y_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_embedded.shape)

X_diagrams = VRP.fit_transform(y_embedded_reshaped)
VRP.plot(X_diagrams)

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in X_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))

print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

Let's start by embedding each normal operation time series to their optimal parameters. This is probably an overkill, but at least we'll see if there is any clear trend in the topological indicators.

In [ ]:
normal_point_clouds_pca, normal_diagrams, normal_entropies, normal_norm_entropies = batch_analyzer(normal_df, 3, 13, 90)


In [ ]:
tmp_entropies = normal_entropies
Entropy_H0 = []
Entropy_H1 = []
tmp_normalized = normal_norm_entropies
Entropy_H0_norm = []
Entropy_H1_norm = []

for item in tmp_entropies:
    Entropy_H0.append(item[0][0])
    Entropy_H1.append(item[0][1])

for item in tmp_normalized:
    Entropy_H0_norm.append(item[0][0])
    Entropy_H1_norm.append(item[0][1])
    
Entropy_H1_series = pd.Series(Entropy_H1)
Entropy_H1_norm_series = pd.Series(Entropy_H1_norm)

print("mean H0 Entropy:", pd.Series(Entropy_H0).mean(), '+/-', pd.Series(Entropy_H0).std())
print("mean H1 Entropy:", pd.Series(Entropy_H1).mean(), '+/-', pd.Series(Entropy_H1).std())
print("mean H0 normalized Entropy:", pd.Series(Entropy_H0_norm).mean(), '+/-', pd.Series(Entropy_H0_norm).std())
print("mean H1 normalized Entropy:", pd.Series(Entropy_H1_norm).mean(), '+/-', pd.Series(Entropy_H1_norm).std())

entropies, ts = plt.subplots(2,figsize=(10,8),sharex = True)

entropies.suptitle('Normal operations dataset persistent entropies')
ts[1].set_xlabel("# timeseries")
ts[0].set_ylabel('Entropy')
ts[1].set_ylabel('normalized  Entropy')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(Entropy_H1,'r-')
ts[0].plot(Entropy_H1_series.rolling(20).mean(), 'g-')
ts[0].plot(Entropy_H0,'b-')

ts[1].set_ylim([-20,40])
ts[1].plot(Entropy_H1_norm,'r-')
ts[1].plot(Entropy_H1_norm_series.rolling(20).mean(), 'g-')
ts[1].plot(Entropy_H0_norm,'b-')

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Normal operations dataset persistent entropies')

data[0].set_xlabel('Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Entropy H$_1$")
data[1].set_ylabel('count')

data[0].set_xlim(6,10)
data[1].set_xlim(0,10)
data[0].hist(Entropy_H0, bins=50)
data[1].hist(Entropy_H1, bins=50)

plt.tight_layout()

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))

stats_entropy.suptitle('Normal operations dataset normalized persistent entropies')

data[0].set_xlabel('Normalized Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Normalized Entropy H$_1$")
data[1].set_ylabel('count')

data[0].set_xlim(-4,10)
data[1].set_xlim(-5,10)
data[0].hist(Entropy_H0_norm, bins=500)
data[1].hist(Entropy_H1_norm, bins=1500)

plt.tight_layout()

there is too much variance in the entropy, better to restrict the analysis to a single set of parameters.

In [ ]:
TE = TakensEmbedding(time_delay=85, dimension=9, stride=3)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PE = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)
Betti = BettiCurve()

normal_point_cloud  = TE.fit_transform(normal_signals)
normal_diagrams = VRP.fit_transform(normal_point_cloud)
normal_entropy = PE.fit_transform(normal_diagrams)
normal_entropynorm = PE_norm.fit_transform(normal_diagrams)
normal_Betti = Betti.fit_transform(normal_diagrams)

In [ ]:
import plotly.express as px
figBetti = px.line()

for i in range(len(normal_Betti[:,0,0])):
#for i in range(50):
    figBetti.add_scatter(y=(normal_Betti[i, 0, :]+5*i))
figBetti.show()

In [ ]:
import statistics as stats
meanBettiH0 = []
medianBettiH0 = []
modeBettiH0 = []
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(normal_Betti[:,0,0])):
    meanBettiH0.append(stats.mean(normal_Betti[i,0,:]))
    medianBettiH0.append(stats.median(normal_Betti[i,0,:]))
    modeBettiH0.append(stats.mode(normal_Betti[i,0,:]))
    
for i in range(len(normal_Betti[:,1,0])):
    meanBettiH1.append(stats.mean(normal_Betti[i,1,:]))
    medianBettiH1.append(stats.median(normal_Betti[i,1,:]))
    modeBettiH1.append(stats.mode(normal_Betti[i,1,:]))
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH0), name="Mean of Betti curve H_0")
figBetti.add_scatter(y=(medianBettiH0), name="Median of Betti curve H_0")
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
#figBetti.add_scatter(y=(modeBettiH0),name="Mode of Betti curve H_0")
figBetti.show()


In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))

stats_entropy.suptitle('Mean of Betti curves for steady state flow')

data[0].set_xlabel('Mean of Betti curve for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Mean of Betti curve for H$_1$")
data[1].set_ylabel('count')

data[0].hist(meanBettiH0, bins=50)
data[1].hist(meanBettiH1)

plt.tight_layout()

In [ ]:
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(normal_Betti[:,1,0])):
    meanBettiH1.append(stats.mean(normal_Betti[i,1,:]))
    medianBettiH1.append(stats.median(normal_Betti[i,1,:]))
    modeBettiH1.append(stats.mode(normal_Betti[i,1,:]))
    
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
figBetti.show()


In [ ]:
pers_H1 = []
pers_H0 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in normal_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for normal BHP')
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")

fig.show() 

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))

stats_entropy.suptitle('Normal operations dataset - Maximum Persistence')

data[0].set_xlabel('Max Persistence for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Max Persistence for H$_1$")
data[1].set_ylabel('count')

data[0].hist(max_pers_H0, bins=50)
data[1].hist(max_pers_H1)

plt.tight_layout()

## Analysis of Slugging P - TPT

In [ ]:
slug_data_single_df = pd.read_csv(DATA_3W_ALL + "/WELL-00001_20170320130025.csv")
signal = slug_data_single_df["P-TPT"]

In [ ]:
optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=optimal_embedding_dimension, time_delay=optimal_time_delay, dimension=6, stride=stride
)

signal_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
signal_embedded_pca = pca.fit_transform(signal_embedded)

plot_point_cloud(signal_embedded_pca)

In [ ]:
find_shortest_file(DATA_3W_ALL, "P-TPT")

In [ ]:
slugging_signals, slugging_df = read_files(DATA_3W_REAL, "P-TPT",3000)

In [ ]:
slugging_df

Convert everything to Bar to get better resolution in the filtration (otherwise the Betti curves are unreadable)

In [ ]:
PatoBar = 1/100000

slugging_df = slugging_df.apply(lambda x: x*PatoBar)
slugging_df 

In [ ]:
slug_point_clouds_pca, slug_diagrams, slug_entropies, slug_norm_entropies = batch_analyzer(slugging_df, 3, 12, 150)

In [ ]:
tmp = slug_entropies
Entropy_H0 = []
Entropy_H1 = []

for item in tmp:
    Entropy_H0.append(item[0][0])
    Entropy_H1.append(item[0][1])

Entropy_H0_slug = pd.Series(Entropy_H0)
Entropy_H1_slug = pd.Series(Entropy_H1)

In [ ]:
plt.plot(Entropy_H0_slug, 'b-')
plt.plot(Entropy_H1_slug, 'r-')
plt.plot(Entropy_H1_slug.rolling(5).mean(), 'g-')

In [ ]:
tmp = slug_norm_entropies
Entropy_H0 = []
Entropy_H1 = []

for item in tmp:
    Entropy_H0.append(item[0][0])
    Entropy_H1.append(item[0][1])

Entropy_H0_slug = pd.Series(Entropy_H0)
Entropy_H1_slug = pd.Series(Entropy_H1)

In [ ]:
plt.plot(Entropy_H0_slug, 'b-')
plt.plot(Entropy_H1_slug, 'r-')
plt.plot(Entropy_H1_slug.rolling(5).mean(), 'g-')

In [ ]:
find_shortest_file(DATA_3W_SIM, "P-TPT")

In [ ]:
slugging_signals, slugging_SIM_df = read_files(DATA_3W_SIM, "P-TPT",3000)
slugging_df = slugging_df.apply(lambda x: x*PatoBar)

In [ ]:
slugSIM_point_clouds_pca, slugSIM_diagrams, slugSIM_entropies, slugSIM_norm_entropies = batch_analyzer(slugging_SIM_df, 2, 20, 90)

In [ ]:
tmp = slugSIM_entropies
Entropy_H0 = []
Entropy_H1 = []

for item in tmp:
    Entropy_H0.append(item[0][0])
    Entropy_H1.append(item[0][1])

Entropy_H0_slug = pd.Series(Entropy_H0)
Entropy_H1_slug = pd.Series(Entropy_H1)

In [ ]:
plt.plot(Entropy_H0_slug, 'b-')
plt.plot(Entropy_H1_slug, 'r-')
plt.plot(Entropy_H1_slug.rolling(5).mean(), 'g-')

In [ ]:
tmp = slugSIM_norm_entropies
Entropy_H0 = []
Entropy_H1 = []

for item in tmp:
    Entropy_H0.append(item[0][0])
    Entropy_H1.append(item[0][1])

Entropy_H0_slug = pd.Series(Entropy_H0)
Entropy_H1_slug = pd.Series(Entropy_H1)

In [ ]:
plt.plot(Entropy_H0_slug, 'b-')
plt.plot(Entropy_H1_slug, 'r-')
plt.plot(Entropy_H1_slug.rolling(5).mean(), 'g-')

In [ ]:
slugging_signals, slugging_df = read_files(DATA_3W_ALL, "P-TPT",3000)
slugging_df = slugging_df.apply(lambda x: x*PatoBar)

In [ ]:
slug_point_clouds_pca, slug_diagrams, slug_entropies, slug_norm_entropies = batch_analyzer(slugging_df, 2, 12, 150)

In [ ]:
slug_data = read_files_bundle(DATA_3W_ALL, "P-TPT",3000)
slug_data = slug_data * PatoBar

VRP_onlyH0 = VietorisRipsPersistence(homology_dimensions =(0,))
VRP_onlyH1 = VietorisRipsPersistence(homology_dimensions =(1,))

TE = TakensEmbedding(time_delay=145, dimension=8, stride=2)
slug_data_norm = TE.fit_transform(slug_data)
slug_data_diagrams_norm = VRP.fit_transform(slug_data_norm)
slug_data_diagrams_H0 = VRP_onlyH0.fit_transform(slug_data_norm)
slug_data_diagrams_H1 = VRP_onlyH1.fit_transform(slug_data_norm)
slug_data_Betti_H0 = Betti.fit_transform(slug_data_diagrams_H0)
slug_data_Betti_H1 = Betti.fit_transform(slug_data_diagrams_H1)

Let's see the results for a simulated slugging event, sample nr.60

In [ ]:
TE.fit_transform_plot(slug_data, sample=60)

In [ ]:
VRP.fit_transform_plot(slug_data_norm, sample=60)

In [ ]:
sample60_H0 = VRP_onlyH0.fit_transform_plot(slug_data_norm, sample=60)

In [ ]:
slug_data_diagrams_H0[60]

In [ ]:
slug_data_Betti_H1[60]

In [ ]:
VRP_onlyH0.fit_transform_plot(slug_data_norm, sample=4)

In [ ]:
Betti.fit_transform_plot(slug_data_diagrams_H0, sample=60)

In [ ]:
slug_data_single_df = pd.read_csv(DATA_3W_ALL + "/SIMULATED_00062.csv")
signal = slug_data_single_df["P-TPT"]
signal = signal / 100000

max_time_delay = 180
max_embedding_dimension = 12
stride = 1

ratio = len(signal)/1500
signal = signal[::int(ratio)]

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print(f"signal lenght: {len(signal)}")
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay


embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
signal_embedded_pca = pca.fit_transform(signal_embedded)
diagram = VRP.fit_transform_plot(signal_embedded_pca[None,:,:])
Betti.fit_transform_plot(diagram)



Let's see sample nr.64 (non simulated)

In [ ]:
TE.fit_transform_plot(slug_data, sample=4)

In [ ]:
VRP.fit_transform_plot(slug_data_norm, sample=4)

In [ ]:
figBettiSlug0 = px.line()

#for i in range(len(slug_data_Betti_H0[:,0,0])):
for i in range(20):
    figBettiSlug0.add_scatter(y=(slug_data_Betti_H0[i, 0, :]+50*i))
figBettiSlug0.show()

In [ ]:
figBettiSlug1 = px.line()

for i in range(len(slug_data_Betti_H1[:,0,0])):
    figBettiSlug1.add_scatter(y=(slug_data_Betti_H1[i, 0, :]+5*i))
figBettiSlug1.show()

In [ ]:
meanBettiH0 = []
medianBettiH0 = []
modeBettiH0 = []
maxBettiH0 = []
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []
maxBettiH1 = []

for i in range(len(slug_data_Betti_H0[:,0,0])):
    meanBettiH0.append(stats.mean(slug_data_Betti_H0[i,0,:]))
    medianBettiH0.append(stats.median(slug_data_Betti_H0[i,0,:]))
    modeBettiH0.append(stats.mode(slug_data_Betti_H0[i,0,:]))
    maxBettiH0.append(np.amax(slug_data_Betti_H0[i,0,:]))
    
for i in range(len(slug_data_Betti_H1[:,0,0])):
    meanBettiH1.append(stats.mean(slug_data_Betti_H1[i,0,:]))
    medianBettiH1.append(stats.median(slug_data_Betti_H1[i,0,:]))
    modeBettiH1.append(stats.mode(slug_data_Betti_H1[i,0,:]))
    maxBettiH1.append(np.amax(slug_data_Betti_H1[i,0,:]))
    
    
figBettiH0 = px.line()
figBettiH0.add_scatter(y=(meanBettiH0), name="Mean of Betti curve H_0")
figBettiH0.add_scatter(y=(medianBettiH0), name="Median of Betti curve H_0")
figBettiH0.add_scatter(y=(modeBettiH0),name="Mode of Betti curve H_0")
#figBetti.add_scatter(y=(maxBettiH0),name="Max of Betti curve H_0")
figBettiH0.show()

In [ ]:
figBettiH1 = px.line()
figBettiH1.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBettiH1.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
figBettiH1.add_scatter(y=(modeBettiH1),name="Mode of Betti curve H_1")
#figBetti.add_scatter(y=(maxBettiH1),name="Max of Betti curve H_1")

figBettiH1.show()

In [ ]:
slug_data_Betti_H0[2]

In [ ]:
maxBettiH0 =[]
maxBettiH1 =[]

for i in range(len(slug_data_Betti_H0[:,0,0])):
    maxBettiH0.append(np.amax(slug_data_Betti_H0[i,0,:]))
for i in range(len(slug_data_Betti_H1[:,0,0])):
    maxBettiH1.append(np.amax(slug_data_Betti_H1[i,0,:]))
        
figBetti = px.line()
#figBetti.add_scatter(y=(maxBettiH0),name="Max of Betti curve H_0")
figBetti.add_scatter(y=(maxBettiH1),name="Max of Betti curve H_1")

figBetti.show()

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in slug_data_diagrams_norm: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for BHP, slugging dataset')
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.show() 

In [ ]:
slug_data_df = pd.read_csv(DATA_3W_SIM + "/SIMULATED_00005.csv")

In [ ]:
slug_data_df

In [ ]:
fig, ts = plt.subplots(8,figsize=(10,20),sharex = True)

fig.suptitle('Slug dataset #1')
ts[7].set_xlabel('time')
ts[0].set_ylabel("P-PDG")
ts[1].set_ylabel('P-TPT')
ts[2].set_ylabel('T-TPT')
ts[3].set_ylabel("P-MON-CKP")
ts[4].set_ylabel('T-JUS-CKP')
ts[5].set_ylabel('P-JUS-CKGL')
ts[6].set_ylabel("T-JUS-CKGL")
ts[7].set_ylabel('QGL')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_data_df.index, slug_data_df["P-PDG"])
ts[1].plot(slug_data_df.index, slug_data_df["P-TPT"])
ts[2].plot(slug_data_df.index, slug_data_df["T-TPT"])
ts[3].plot(slug_data_df.index, slug_data_df["P-MON-CKP"])
ts[4].plot(slug_data_df.index, slug_data_df["T-JUS-CKP"])
ts[5].plot(slug_data_df.index, slug_data_df["P-JUS-CKGL"])
ts[6].plot(slug_data_df.index, slug_data_df["T-JUS-CKGL"])
ts[7].plot(slug_data_df.index, slug_data_df["QGL"])


In [ ]:
slug_data_df['P-TPT'].isnull().values.any()

In [ ]:
slug_data_df= slug_data_df.interpolate()

In [ ]:
period = 50
periodicSampler = Resampler(period=period)
slug_data_df = slug_data_df.interpolate(method='linear')
signal = slug_data_df["P-TPT"]

#slug_data_df.index, slug_data_df["P-TPT"]
index_sampled, signal_sampled = periodicSampler.fit_transform_resample(slug_data_df.index, signal)

#print(index_sampled, signal_sampled, len(signal_sampled))

In [ ]:
signal_downsampled = pd.DataFrame(signal_sampled,
                  index=index_sampled, columns=['P-TPT'])
#signal_downsampled['P-TPT']
plt.xlabel('time');
plt.ylabel('P-TPT');
plt.plot(signal_downsampled.index,signal_downsampled['P-TPT'], 'bo-')

In [ ]:
print('length of signal to analyze', len(signal_downsampled))

max_time_delay = 150 
max_embedding_dimension = 15
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal_downsampled, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded = embedder.fit_transform(signal_downsampled)

pca = PCA(n_components=3)
signal_embedded_pca = pca.fit_transform(signal_embedded)

plot_point_cloud(signal_embedded_pca)

In [ ]:
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

signal_embedded_reshaped = signal_embedded.reshape(1, *signal_embedded.shape)
print(f"y_wn_embedded_reshaped.shape",signal_embedded_reshaped.shape)
print(f"y_wn_embedded.shape", signal_embedded.shape)

PerHom_signal = VRP.fit_transform(signal_embedded_reshaped)
VRP.plot(PerHom_signal)

In [ ]:
PE_slug = PersistenceEntropy()
PE_slug_norm = PersistenceEntropy(normalize=True)
features = PE_slug.fit_transform(PerHom_signal)
features_norm = PE_slug_norm.fit_transform(PerHom_signal)

In [ ]:
features

In [ ]:
features_norm

In [ ]:
slug_data_sim_df = pd.read_csv(DATA_3W_NORMAL + "/WELL-00021_20170509013517.csv")

In [ ]:
slug_data_sim_df = slug_data_sim_df.head(18000)

fig, ts = plt.subplots(8,figsize=(10,20),sharex = True)

fig.suptitle('Simulated Slug dataset #1')
ts[7].set_xlabel('time')
ts[0].set_ylabel("P-PDG")
ts[1].set_ylabel('P-TPT')
ts[2].set_ylabel('T-TPT')
ts[3].set_ylabel("P-MON-CKP")
ts[4].set_ylabel('T-JUS-CKP')
ts[5].set_ylabel('P-JUS-CKGL')
ts[6].set_ylabel("T-JUS-CKGL")
ts[7].set_ylabel('QGL')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(slug_data_sim_df.index, slug_data_sim_df["P-PDG"])
ts[1].plot(slug_data_sim_df.index, slug_data_sim_df["P-TPT"])
ts[2].plot(slug_data_sim_df.index, slug_data_sim_df["T-TPT"])
ts[3].plot(slug_data_sim_df.index, slug_data_sim_df["P-MON-CKP"])
ts[4].plot(slug_data_sim_df.index, slug_data_sim_df["T-JUS-CKP"])
ts[5].plot(slug_data_sim_df.index, slug_data_sim_df["P-JUS-CKGL"])
ts[6].plot(slug_data_sim_df.index, slug_data_sim_df["T-JUS-CKGL"])
ts[7].plot(slug_data_sim_df.index, slug_data_sim_df["QGL"])


In [ ]:
period = 5
periodicSampler = Resampler(period=period)
signal = slug_data_sim_df["P-TPT"]

slug_data_sim_df.index, slug_data_sim_df["P-TPT"]
index_sampled, signal_sim_sampled = periodicSampler.fit_transform_resample(slug_data_sim_df.index, signal)


In [ ]:
signal_sim_sampled

In [ ]:
signal_sim_downsampled = pd.DataFrame(signal_sim_sampled,
                  index=index_sampled, columns=['P-TPT'])
plt.xlabel('time');
plt.ylabel('P-TPT');
plt.plot(signal_sim_downsampled.index,signal_sim_downsampled['P-TPT'], 'bo-')

In [ ]:
print('length of signal to analyze', len(signal_sim_downsampled))

max_time_delay = 50 
max_embedding_dimension = 10
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal_sim_downsampled, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_sim_embedded = embedder.fit_transform(signal_sim_downsampled)

pca = PCA(n_components=3)
signal_sim_embedded_pca = pca.fit_transform(signal_sim_embedded)

plot_point_cloud(signal_sim_embedded_pca)

In [ ]:
signal_sim_embedded_reshaped = signal_sim_embedded.reshape(1, *signal_sim_embedded.shape)
print(f"y_wn_embedded_reshaped.shape",signal_sim_embedded_reshaped.shape)
print(f"y_wn_embedded.shape", signal_sim_embedded_reshaped.shape)

PerHom_sim_signal = VRP.fit_transform(signal_sim_embedded_reshaped)
features_sim = PE_slug.fit_transform(PerHom_signal)
features_sim_norm = PE_slug_norm.fit_transform(PerHom_signal)

In [ ]:
features_sim

In [ ]:
features_sim_norm

In [ ]:
VRP.plot(PerHom_sim_signal)

In [ ]:
PerHom_sim_signal.shape

In [ ]:
PerHom_signal.shape

#tmp = np.vstack([PerHom_signal,PerHom_sim_signal])
#tmp.shape

In [ ]:
diagramScaler = Scaler()
PerHom_signal_scaled = diagramScaler.fit_transform(PerHom_signal)
diagramScaler.plot(PerHom_signal_scaled)

In [ ]:
PerHom_signal_sim_scaled = diagramScaler.fit_transform(PerHom_sim_signal)
diagramScaler.plot(PerHom_signal_sim_scaled)

In [ ]:
features_sim = PE_slug.fit_transform(PerHom_signal_scaled)
features_sim_norm = PE_slug_norm.fit_transform(PerHom_signal_scaled)
features_sim

In [ ]:
features_sim_norm

In [ ]:
HK = HeatKernel(sigma=.15, n_bins=60, n_jobs=-1) 
HK_slug = HK.fit_transform_plot(PerHom_signal_scaled, homology_dimension_idx=1)

In [ ]:
HK.fit_transform_plot(PerHom_signal_sim_scaled, homology_dimension_idx=1)

In [ ]:
DATA_3W_NORMAL + "/WELL-00006_20170207180042.csv", "P-TPT",3000)